# HuggingFace counts 

In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import geopandas
import matplotlib.pyplot as plt
from shapely.geometry import Point

In [ ]:
html_text = requests.get('https://huggingface.co/languages').text

In [ ]:
soup = BeautifulSoup(html_text, 'html.parser')

In [ ]:
rows = soup.find_all('tr')

# Initialize lists to store data
languages = []
iso_codes = []
datasets_count = []
models_count = []

In [ ]:
for row in rows:
    cells = row.find_all('td')
    if len(cells) > 0:  # Exclude header row
        languages.append(cells[0].get_text(strip=True))
        iso_codes.append(cells[1].find('code').get_text(strip=True))
        datasets_count.append(int(cells[2].get_text(strip=True).replace(',', '')))
        models_count.append(int(cells[3].get_text(strip=True).replace(',', '')))

In [ ]:
df = pd.DataFrame({
    'Language': languages,
    'ISO Code': iso_codes,
    'Datasets Count': datasets_count,
    'Models Count': models_count
})

In [ ]:
df.sort_values(by="Datasets Count", ascending=False).loc[:40]

In [ ]:
datasets = df.sort_values(by="Datasets Count", ascending=False).loc[:, "Datasets Count"].values

In [ ]:
plt.semilogy(datasets[1:], ".")

In [ ]:
n_of_speakers_data = pd.read_csv('data/data.tsv', sep='\t')
glottolog_data = pd.read_csv('data/languages_and_dialects_geo.csv')
worldmap = geopandas.read_file(geopandas.datasets.get_path("naturalearth_lowres"))

In [ ]:
huggingface_n_speakers_all = pd.merge(df, n_of_speakers_data, left_on = 'ISO Code', right_on = 'ISO 639 code', how = 'outer')

In [ ]:
huggingface_n_speakers = huggingface_n_speakers_all.dropna(subset=['Language', 'Number of speakers (rounded)'])

In [ ]:
huggingface_language_per_speaker = huggingface_n_speakers['Number of speakers (rounded)'].value_counts()

In [ ]:
huggingface_language_per_speaker

In [ ]:
huggingface_language_per_speaker.index = huggingface_language_per_speaker.index.str.replace('<10K', '5000').str.replace('10K-100K', '50000').astype(int)

In [ ]:
huggingface_language_per_speaker

In [ ]:
huggingface_language_per_speaker_sorted = huggingface_language_per_speaker.sort_index()

In [ ]:
huggingface_language_per_speaker_sorted.plot(kind='bar', figsize=(20,8))
plt.xlabel('Number of Speakers (Rounded)')
plt.ylabel('Frequency')
plt.title('Distribution of Speakers per language')
plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
plt.show()

# Not in huggingface, with speaker counts

In [ ]:
n_speakers_no_huggingface = huggingface_n_speakers_all[huggingface_n_speakers_all['Language'].isna()]
n_speakers_no_huggingface_values = n_speakers_no_huggingface['Number of speakers (rounded)'].value_counts()
n_speakers_no_huggingface_values.index = n_speakers_no_huggingface_values.index.str.replace('<10K', '5000').str.replace('10K-100K', '50000').astype(int)
n_speakers_no_huggingface_values_sorted = n_speakers_no_huggingface_values.sort_index()
#there is some problem with the values here...
n_speakers_no_huggingface_values_sorted = n_speakers_no_huggingface_values_sorted[~n_speakers_no_huggingface_values_sorted.index.isin([10000, 80000])]
n_speakers_no_huggingface_values_sorted = n_speakers_no_huggingface_values_sorted[~((n_speakers_no_huggingface_values_sorted.index == 50000) & (n_speakers_no_huggingface_values_sorted.values == 1))]

In [ ]:
n_speakers_no_huggingface_values_sorted.plot(kind='bar', figsize=(20,8))
plt.xlabel('Number of Speakers (Rounded)')
plt.ylabel('Frequency')
plt.title('Distribution of Speakers per language')
plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
plt.show()

In [ ]:
n_speakers_no_huggingface_values_sorted.plot(kind='bar', figsize=(20, 8), color='blue', position=0.5, width=0.4, label='HF_NOT_covered')

huggingface_language_per_speaker_sorted.plot(kind='bar', figsize=(20, 8), color='orange', position=-0.5, width=0.4, label='HF_covered')

plt.xlabel('Number of Speakers (Rounded)')
plt.ylabel('Frequency')
plt.title('Distribution of Speakers per language')
plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Assuming n_speakers_no_huggingface_values_sorted1 and n_speakers_no_huggingface_values_sorted2 are your Series objects containing the distributions

# Align the two series
aligned_series = n_speakers_no_huggingface_values_sorted.align(huggingface_language_per_speaker_sorted, fill_value=0)

# Plot histogram for the aligned distributions, stacking them
ax = aligned_series[0].plot(kind='bar', figsize=(20, 8), color='blue', label='HF_NOT_covered')
aligned_series[1].plot(kind='bar', color='orange', label='HF_covered', ax=ax, bottom=aligned_series[0])

plt.xlabel('Number of Speakers (Rounded)')
plt.ylabel('Frequency')
plt.title('Distribution of Speakers per language')
plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
plt.legend()
plt.show()

# HuggingFace Coverage

In [ ]:
huggingface_language_locaysh = pd.merge(df, glottolog_data, left_on = 'ISO Code', right_on = 'isocodes', how = 'outer')
huggingface_language_locaysh = huggingface_language_locaysh.dropna(subset=['ISO Code'])
huggingface_language_locaysh = huggingface_language_locaysh.dropna(subset=['latitude', 'longitude'])

In [ ]:
huggingface_language_locaysh

In [ ]:
points = huggingface_language_locaysh.apply(lambda r: Point(r.longitude, r.latitude), axis=1)
gdb = geopandas.GeoDataFrame(huggingface_language_locaysh.assign(geometry=points), \
                        crs={'init' :'epsg:4326'})
xys = gdb['geometry'].apply(lambda pt: pd.Series({'x': pt.x, 'y': pt.y}))
gdb['x'] = xys['x']
gdb['y'] = xys['y']

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
worldmap.plot(color="white", edgecolor = 'black',ax=ax, linewidth=0.3)
gdb.plot(ax=ax, color = 'blue', markersize = 5)
plt.show()

# Glottolog Coverage

In [ ]:
glottolog_data_locays = glottolog_data.dropna(subset=['latitude', 'longitude'])
glottolog_data_locays = glottolog_data_locays[glottolog_data_locays['level'] != 'dialect']

In [ ]:
points = glottolog_data_locays.apply(lambda r: Point(r.longitude, r.latitude), axis=1)
gdb = geopandas.GeoDataFrame(glottolog_data_locays.assign(geometry=points), \
                        crs={'init' :'epsg:4326'})
xys = gdb['geometry'].apply(lambda pt: pd.Series({'x': pt.x, 'y': pt.y}))
gdb['x'] = xys['x']
gdb['y'] = xys['y']

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
worldmap.plot(color="white", edgecolor = 'black',ax=ax, linewidth=0.3)
gdb.plot(ax=ax, color = 'blue', markersize = 5)
plt.show()